In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import pandas as pd
import xskillscore as xs
import matplotlib.pyplot as plt
import numpy as np

import albedo_functions as af

from pathlib import Path

import concurrent.futures

import logging
import psutil

In [ ]:
# Configura il logging
log_file = "seasonal_ACC_calculation.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
#        logging.StreamHandler()  # Still see some output in Jupyter
    ]
)

# Percorso ai dati
DATA_PATH = POST_DATA
OUTPUT_PATH = FIG_DIR_2025
OBS_PATH = WORK_DIR

# Esperimenti e variabili
exp_sens = "a52o"
exp_ctrl = "a1ua"
var = "tas"

In [ ]:
season = 'DJF'
y1=3
y2=4
lead = f"{y1}-{y2}"
lead_number = y2 - y1 + 1

In [ ]:
# Caricamento dataset di controllo
dset_ctrl_path = f"{DATA_PATH}/{exp_ctrl}/1x1/{var}/{exp_ctrl}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_m{season}_rad.nc"
dset_ctrl = xr.open_dataset(dset_ctrl_path, chunks={'time':-1})
if var == 'snd':
    dset_ctrl['lon']=dset_ctrl['lon']%360
    dset_ctrl_ = dset_ctrl.sortby('lon')
    dset_ctrl = dset_ctrl_
dset_ctrl['time'] = pd.to_datetime(dset_ctrl['time'].values).normalize().to_period('M').start_time

# Caricamento dataset sensibile
dset_sens_path = f"{DATA_PATH}/{exp_sens}/1x1/{var}/{exp_sens}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_m{season}_rad.nc"
dset_sens = xr.open_dataset(dset_sens_path, chunks={'time':-1})
dset_sens['time'] = pd.to_datetime(dset_sens['time'].values).normalize().to_period('M').start_time

if season == 'DJF':
    dset_ctrl['time'] = [pd.Timestamp(t).replace(month=12) for t in dset_ctrl['time'].values]
    dset_sens['time'] = [pd.Timestamp(t).replace(month=12) for t in dset_sens['time'].values]

# Caricamento osservazioni
if var=='alb':
    obs_path = OBS_PATH / f"GLASS_1x1_{lead_number}{season}.nc"
elif var == 'tas':
    obs_path = OBS_PATH / f"ERA5_tas_1x1_{lead_number}{season}.nc"
elif var == 'sd':
    obs_path = OBS_PATH / f"ERA5_sd_1x1_{lead_number}{season}.nc"
elif var == 'snd':
    obs_path = OBS_PATH / f"ERA5_snd_1x1_{lead_number}{season}.nc"
    
obs = xr.open_dataset(obs_path, chunks={'time':-1})
obs['time'] = pd.to_datetime(obs['time'].values).normalize().to_period('M').start_time
if var=='alb':
    obs = obs.rename({'bb_albedo': 'alb'})
elif var == 'tas':
    obs = obs.rename({'2t': 'tas'})

obs = obs.sel(time=slice(dset_ctrl.time[6], dset_ctrl.time[-1]))
dset_sens = dset_sens.sel(time=slice(dset_ctrl.time[6], dset_ctrl.time[-1]))
dset_ctrl = dset_ctrl.sel(time=slice(dset_ctrl.time[6], dset_ctrl.time[-1]))

In [ ]:
if var=='alb':
    # Sostituzione dei valori non validi (< 0) con NaN
    obs_ = xr.where(obs < 0, np.nan, obs)
    dset_ctrl_ = xr.where(dset_ctrl < 0, np.nan, dset_ctrl)
    dset_sens_ = xr.where(dset_sens < 0, np.nan, dset_sens)
    # Ulteriore filtraggio: rimuove dati con deviazione standard bassa
    obs = xr.where(obs_.std('time') >= 0.001, obs_, np.nan)
    dset_ctrl = xr.where(dset_ctrl_.std('time') >= 0.001, dset_ctrl_, np.nan)
    dset_sens = xr.where(dset_sens_.std('time') >= 0.005, dset_sens_, np.nan)

In [ ]:
# Calcolo anomalie
dset_ctrl_anom = dset_ctrl - dset_ctrl.mean('time')

# Adjust longitude values
#dset_ctrl_anom['lon'] = dset_ctrl_anom['lon'] % 360 - 0.5
#dset_ctrl_anom = dset_ctrl_anom.sortby("lon")

In [ ]:
dset_sens_anom = dset_sens - dset_sens.mean('time')
obs_anom = obs - obs.mean('time')

In [ ]:
# Calcolo correlazioni
dset_ctrl_acc, dset_ctrl_p = af.alb_corr(obs_anom[f"{var}"], dset_ctrl_anom[f"{var}"])
dset_sens_acc, dset_sens_p = af.alb_corr(obs_anom[f"{var}"], dset_sens_anom[f"{var}"])

# Conversione in Dataset
dset_ctrl_acc_ = dset_ctrl_acc.to_dataset(name=f"{var}")
dset_ctrl_p_ = dset_ctrl_p.to_dataset(name=f"{var}")
    
dset_sens_acc_ = dset_sens_acc.to_dataset(name=f"{var}")
dset_sens_p_ = dset_sens_p.to_dataset(name=f"{var}")

sig_delta = af.bootstrap_quantile_correlation(dset_sens_anom[f"{var}"], dset_ctrl_anom[f"{var}"], obs_anom[f"{var}"].transpose('lat','lon','time'))
sig_delta_ = sig_delta.to_dataset(name=f"{var}")

In [ ]:
#salva file
if var == 'alb':
    dset_ctrl_acc_.to_netcdf(f"{OBS_PATH}/{exp_ctrl}_albedo_ACC_r_{lead}_{season}.nc")
    dset_ctrl_p_.to_netcdf(f"{OBS_PATH}/{exp_ctrl}_albedo_ACC_p_{lead}_{season}.nc")
    dset_sens_acc_.to_netcdf(f"{OBS_PATH}/{exp_sens}_albedo_ACC_r_{lead}_{season}.nc")
    dset_sens_p_.to_netcdf(f"{OBS_PATH}/{exp_sens}_albedo_ACC_p_{lead}_{season}.nc")
    sig_delta_.to_netcdf(f"{OBS_PATH}/{exp_sens}-{exp_ctrl}_albedo_bootstrap_{lead}_{season}.nc")
elif var == 'tas':
    dset_ctrl_acc_.to_netcdf(f"{OBS_PATH}/{exp_ctrl}_tas_ACC_r_{lead}_{season}_1999_oce.nc")
    dset_ctrl_p_.to_netcdf(f"{OBS_PATH}/{exp_ctrl}_tas_ACC_p_{lead}_{season}_1999_oce.nc")
    dset_sens_acc_.to_netcdf(f"{OBS_PATH}/{exp_sens}_tas_ACC_r_{lead}_{season}_1999_oce.nc")
    dset_sens_p_.to_netcdf(f"{OBS_PATH}/{exp_sens}_tas_ACC_p_{lead}_{season}_1999_oce.nc")
    sig_delta_.to_netcdf(f"{OBS_PATH}/{exp_sens}-{exp_ctrl}_tas_bootstrap_{lead}_{season}_1999_oce.nc")
elif var == 'snd':
    dset_ctrl_acc_.to_netcdf(f"{OBS_PATH}/{exp_ctrl}_snd_ACC_r_{lead}_{season}.nc")
    dset_ctrl_p_.to_netcdf(f"{OBS_PATH}/{exp_ctrl}_snd_ACC_p_{lead}_{season}.nc")
    dset_sens_acc_.to_netcdf(f"{OBS_PATH}/{exp_sens}_snd_ACC_r_{lead}_{season}.nc")
    dset_sens_p_.to_netcdf(f"{OBS_PATH}/{exp_sens}_snd_ACC_p_{lead}_{season}.nc")
    sig_delta_.to_netcdf(f"{OBS_PATH}/{exp_sens}-{exp_ctrl}_snd_bootstrap_{lead}_{season}.nc")